In [9]:
import os
import io
from IPython.display import Image, display, HTML
from PIL import Image
import base64 
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file
hf_api_key = os.environ['HF_API_KEY']

In [10]:
# Helper function
import requests, json

#Summarization endpoint
def get_completion(inputs, parameters=None,ENDPOINT_URL=os.environ['HF_API_SUMMARY_BASE']): 
    headers = {
      "Authorization": f"Bearer {hf_api_key}",
      "Content-Type": "application/json"
    }
    data = { "inputs": inputs }
    if parameters is not None:
        data.update({"parameters": parameters})
    response = requests.request("POST",
                                ENDPOINT_URL, headers=headers,
                                data=json.dumps(data)
                               )
    return json.loads(response.content.decode("utf-8"))

In [11]:
API_URL = os.environ['HF_API_NER_BASE'] #NER endpoint
text = "My name is Reshmithaa, I'm building DeepLearningAI and I live in India"
get_completion(text, parameters=None, ENDPOINT_URL= API_URL)

[{'entity': 'B-PER',
  'score': 0.99681157,
  'index': 4,
  'word': 'Re',
  'start': 11,
  'end': 13},
 {'entity': 'B-PER',
  'score': 0.98521936,
  'index': 5,
  'word': '##sh',
  'start': 13,
  'end': 15},
 {'entity': 'I-PER',
  'score': 0.93017405,
  'index': 6,
  'word': '##mith',
  'start': 15,
  'end': 19},
 {'entity': 'I-PER',
  'score': 0.89677936,
  'index': 7,
  'word': '##aa',
  'start': 19,
  'end': 21},
 {'entity': 'B-ORG',
  'score': 0.98561066,
  'index': 13,
  'word': 'Deep',
  'start': 36,
  'end': 40},
 {'entity': 'I-ORG',
  'score': 0.99251324,
  'index': 14,
  'word': '##L',
  'start': 40,
  'end': 41},
 {'entity': 'I-ORG',
  'score': 0.9915309,
  'index': 15,
  'word': '##ear',
  'start': 41,
  'end': 44},
 {'entity': 'I-ORG',
  'score': 0.9918094,
  'index': 16,
  'word': '##ning',
  'start': 44,
  'end': 48},
 {'entity': 'I-ORG',
  'score': 0.839178,
  'index': 17,
  'word': '##A',
  'start': 48,
  'end': 49},
 {'entity': 'I-ORG',
  'score': 0.6118238,
  'index':

In [12]:
import gradio as gr
def ner(input):
    output = get_completion(input, parameters=None, ENDPOINT_URL=API_URL)
    return {"text": input, "entities": output}

gr.close_all()
demo = gr.Interface(fn=ner,
                    inputs=[gr.Textbox(label="Text to find entities", lines=2)],
                    outputs=[gr.HighlightedText(label="Text with entities")],
                    title="NER with dslim/bert-base-NER",
                    description="Find entities using the `dslim/bert-base-NER` model under the hood!",
                    allow_flagging="never",
                    #Here we introduce a new tag, examples, easy to use examples for your application
                    examples=["My name is Reshmithaa and I live in Chennai", "My name is Poli and work at HuggingFace"])
demo.launch(share=True, server_port=int(os.environ['PORT3']))

Closing server running on port: 7862
Closing server running on port: 7863
Closing server running on port: 7863
Closing server running on port: 7862
Running on local URL:  https://0.0.0.0:7862
IMPORTANT: You are using gradio version 3.37.0, however version 4.44.1 is available, please upgrade.
--------

Could not create share link. Missing file: /usr/local/lib/python3.9/site-packages/gradio/frpc_linux_amd64_v0.2. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64
2. Rename the downloaded file to: frpc_linux_amd64_v0.2
3. Move the file to this location: /usr/local/lib/python3.9/site-packages/gradio


In [13]:
def merge_tokens(tokens):
    merged_tokens = []
    for token in tokens:
        if merged_tokens and token['entity'].startswith('I-') and merged_tokens[-1]['entity'].endswith(token['entity'][2:]):
            # If current token continues the entity of the last one, merge them
            last_token = merged_tokens[-1]
            last_token['word'] += token['word'].replace('##', '')
            last_token['end'] = token['end']
            last_token['score'] = (last_token['score'] + token['score']) / 2
        else:
            # Otherwise, add the token to the list
            merged_tokens.append(token)

    return merged_tokens

def ner(input):
    output = get_completion(input, parameters=None, ENDPOINT_URL=API_URL)
    merged_tokens = merge_tokens(output)
    return {"text": input, "entities": merged_tokens}

gr.close_all()
demo = gr.Interface(fn=ner,
                    inputs=[gr.Textbox(label="Text to find entities", lines=2)],
                    outputs=[gr.HighlightedText(label="Text with entities")],
                    title="NER with dslim/bert-base-NER",
                    description="Find entities using the `dslim/bert-base-NER` model under the hood!",
                    allow_flagging="never",
                    examples=["My name is Reshmithaa, I'm building DeeplearningAI and I live in Chennai", "My name is Poli, I live in Vienna and work at HuggingFace"])

demo.launch(share=True, server_port=int(os.environ['PORT4']))

Closing server running on port: 7863
Closing server running on port: 7862
Closing server running on port: 7862
Closing server running on port: 7862
Closing server running on port: 7863
Running on local URL:  https://0.0.0.0:7863
IMPORTANT: You are using gradio version 3.37.0, however version 4.44.1 is available, please upgrade.
--------

Could not create share link. Missing file: /usr/local/lib/python3.9/site-packages/gradio/frpc_linux_amd64_v0.2. 

Please check your internet connection. This can happen if your antivirus software blocks the download of this file. You can install manually by following these steps: 

1. Download this file: https://cdn-media.huggingface.co/frpc-gradio-0.2/frpc_linux_amd64
2. Rename the downloaded file to: frpc_linux_amd64_v0.2
3. Move the file to this location: /usr/local/lib/python3.9/site-packages/gradio


In [14]:
gr.close_all()

Closing server running on port: 7863
Closing server running on port: 7862
Closing server running on port: 7862
Closing server running on port: 7863
Closing server running on port: 7862
Closing server running on port: 7863
